# AI Supply Chain Optimization Agent

Demand forecasting, inventory planning, supplier scoring, disruption simulation, and explainable replenishment recommendations. Uses synthetic data only. Human review is required; this prototype does not place orders or connect to real ERP systems.

In [ ]:
import math, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=180, freq="D")
products = [
    ("SKU-1001", "Wireless Mouse", 28, 0.012, 450, 8),
    ("SKU-1002", "USB-C Hub", 18, 0.008, 1250, 12),
    ("SKU-1003", "Laptop Stand", 14, 0.004, 900, 10),
    ("SKU-1004", "Keyboard", 22, -0.002, 780, 7),
]
records = []
for sku, name, base, trend, cost, lead in products:
    for i, day in enumerate(dates):
        weekly = 1 + 0.15 * np.sin(2 * np.pi * day.dayofweek / 7)
        qty = max(0, int(round(base * weekly * (1 + trend*i) + rng.normal(0, base*0.15))))
        records.append({"date": day, "sku": sku, "product": name, "demand_units": qty})
demand = pd.DataFrame(records)
demand.head()

## 1. Inventory and supplier data

In [ ]:
inventory = pd.DataFrame([
    {"sku":"SKU-1001","on_hand":190,"on_order":40,"unit_cost":450,"lead_time_days":8,"z":1.65},
    {"sku":"SKU-1002","on_hand":95,"on_order":0,"unit_cost":1250,"lead_time_days":12,"z":1.65},
    {"sku":"SKU-1003","on_hand":220,"on_order":30,"unit_cost":900,"lead_time_days":10,"z":1.65},
    {"sku":"SKU-1004","on_hand":80,"on_order":50,"unit_cost":780,"lead_time_days":7,"z":1.65},
])
suppliers = pd.DataFrame([
    ["Northstar Components","SKU-1001",8,.96,.012,.12,440],
    ["Metro Wholesale","SKU-1001",6,.91,.025,.20,455],
    ["Apex Electronics","SKU-1002",12,.89,.018,.32,1210],
    ["Vertex Supply","SKU-1002",10,.94,.015,.16,1270],
    ["Greenline Products","SKU-1003",10,.95,.010,.14,880],
    ["Summit Trading","SKU-1004",7,.92,.022,.18,760],
], columns=["supplier","sku","lead_time_days","on_time_rate","defect_rate","disruption_risk","unit_price"])
display(inventory)
display(suppliers)

## 2. Forecast demand and calculate reorder plans
Moving average is an explainable baseline. Backtest alternatives before operational use.

In [ ]:
def make_forecast(history, window=28):
    out = []
    for sku, group in history.groupby("sku"):
        recent = group.sort_values("date").tail(window).demand_units
        out.append({"sku":sku, "avg_daily_demand":recent.mean(),
                    "daily_std":recent.std(), "forecast_30d":math.ceil(recent.mean()*30)})
    return pd.DataFrame(out)

forecast = make_forecast(demand)
plan = inventory.merge(forecast, on="sku")
plan["safety_stock"] = (plan.z * plan.daily_std * np.sqrt(plan.lead_time_days)).fillna(0).apply(math.ceil)
plan["lead_time_demand"] = (plan.avg_daily_demand * plan.lead_time_days).apply(math.ceil)
plan["reorder_point"] = plan.safety_stock + plan.lead_time_demand
plan["inventory_position"] = plan.on_hand + plan.on_order
plan["target_stock"] = plan.avg_daily_demand * 30 + plan.safety_stock
plan["suggested_order_qty"] = np.maximum(0, np.ceil(plan.target_stock-plan.inventory_position)).astype(int)
plan["reorder_required"] = plan.inventory_position <= plan.reorder_point
plan["estimated_order_value"] = plan.suggested_order_qty * plan.unit_cost
display(plan[["sku","avg_daily_demand","safety_stock","reorder_point","inventory_position","reorder_required","suggested_order_qty","estimated_order_value"]])

## 3. Supplier performance scoring
Higher heuristic score indicates a more favorable supplier. Weights are illustrative and should be calibrated to historical performance.

In [ ]:
def rank_suppliers(df):
    scored = df.copy()
    max_lead = max(scored.lead_time_days.max(), 1)
    scored["score"] = (40*scored.on_time_rate + 25*(1-scored.defect_rate)
                       + 20*(1-scored.disruption_risk)
                       + 15*(1-scored.lead_time_days/max_lead))
    return scored.sort_values(["sku","score"], ascending=[True,False])

supplier_ranking = rank_suppliers(suppliers)
display(supplier_ranking)

## 4. Supplier disruption what-if simulation

In [ ]:
def simulate_delay(sku, extra_days=7):
    row = plan.loc[plan.sku.eq(sku)].iloc[0]
    supplier = supplier_ranking.loc[supplier_ranking.sku.eq(sku)].iloc[0]
    extended = int(supplier.lead_time_days + extra_days)
    demand_during_lead = float(row.avg_daily_demand * extended)
    gap = max(0, math.ceil(demand_during_lead + row.safety_stock - row.inventory_position))
    return {
        "sku":sku, "supplier":supplier.supplier,
        "base_lead_days":int(supplier.lead_time_days),
        "extra_delay_days":extra_days, "extended_lead_days":extended,
        "estimated_demand_during_lead":round(demand_during_lead,1),
        "estimated_coverage_gap_units":gap,
        "risk_level":"High" if gap else "Lower in this scenario"
    }

scenario = simulate_delay("SKU-1002", 7)
scenario

## 5. Explainable planning agent recommendations
These are decision-support suggestions only. No purchase orders are created.

In [ ]:
def recommend(plan_df, supplier_df, scenario):
    result = []
    for _, row in plan_df.iterrows():
        options = supplier_df[supplier_df.sku.eq(row.sku)]
        preferred = options.iloc[0].supplier if len(options) else None
        if row.reorder_required and row.suggested_order_qty > 0:
            action = f"Review replenishment of {int(row.suggested_order_qty)} units."
            priority, reason = "High", "Inventory position is at or below reorder point."
        else:
            action = "Monitor stock and refresh forecast."
            priority, reason = "Inventory position is currently above reorder point."
        result.append({
            "sku":row.sku, "priority":priority, "recommendation":action,
            "reason":reason, "supplier_candidate":preferred,
            "estimated_order_value":round(float(row.estimated_order_value),2),
            "human_approval_required":True
        })
    result.append({
        "sku":scenario["sku"], "priority":scenario["risk_level"],
        "recommendation":"Review contingency sourcing and safety-stock options.",
        "reason":f"Delay scenario estimates a coverage gap of {scenario['estimated_coverage_gap_units']} units.",
        "supplier_candidate":None, "estimated_order_value":None,
        "human_approval_required":True
    })
    return result

recommendations = recommend(plan, supplier_ranking, scenario)
print(json.dumps(recommendations, indent=2))

## 6. Visualize demand and inventory

In [ ]:
sku = "SKU-1002"
subset = demand[demand.sku.eq(sku)].sort_values("date")
plt.figure(figsize=(11,4))
plt.plot(subset.date, subset.demand_units)
plt.title(f"Daily demand — {sku}")
plt.xlabel("Date"); plt.ylabel("Units"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
plt.bar(plan.sku, plan.inventory_position, label="Inventory position")
plt.plot(plan.sku, plan.reorder_point, marker="o", label="Reorder point")
plt.title("Inventory position vs reorder point")
plt.ylabel("Units"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Export results

In [ ]:
out = Path("supply_chain_outputs")
out.mkdir(exist_ok=True)
plan.to_csv(out/"inventory_recommendations.csv", index=False)
supplier_ranking.to_csv(out/"supplier_rankings.csv", index=False)
(out/"agent_recommendations.json").write_text(json.dumps(recommendations, indent=2), encoding="utf-8")
(out/"disruption_scenario.json").write_text(json.dumps(scenario, indent=2), encoding="utf-8")
print("Saved output files in", out.resolve())

## 8. Production-readiness checklist

- Backtest forecasts with rolling time-based splits and compare against seasonal-naive/statistical/ML baselines.
- Validate minimum order quantities, pack sizes, supplier contracts, lead-time distributions, and capacity constraints.
- Track forecast error, service level, stockouts, inventory turns, supplier OTIF, and total landed cost.
- Account for promotions, holidays, returns, stockouts, and new-product cold starts.
- Integrate ERP/WMS systems through least-privilege, audited interfaces.
- Require human approval for purchasing and supplier changes.

**Limitations:** All data and scoring weights are synthetic/illustrative. The moving-average forecast is not production-validated. This notebook does not execute procurement or guarantee stock availability.